# Clean up your SageMaker AI environment

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.6.0</strong> and the <strong>SageMaker Python SDK version 3.27.0</strong>.
</div>

This notebook deletes the resources that the workshop notebooks created. Each cell asks for confirmation before it deletes anything. The cells:
- delete the SageMaker projects, their CloudFormation stacks, and their S3 buckets
- delete the inference endpoints
- delete the monitoring pipeline, its schedule, and its SNS topic from step 4
- delete the ML pipeline from step 3 and the model package group with all model versions
- delete the workshop data under the workshop S3 prefix, and the project template and seed code from step 5
- delete the MLflow app and the container image repository in Amazon ECR
- delete the feature groups of the optional notebook 03c

<div class="alert alert-warning">The lists of projects, feature groups, and endpoints show <b>all</b> resources in the account and Region, also the resources that you created outside this workshop. Check each name before you confirm.</div>

<div class="alert alert-info">You don't need to run this notebook if you use an AWS-provided account in an instructor-led workshop.</div>

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), (
    f"This workshop needs Python 3.12, but the kernel runs Python {sys.version_info.major}.{sys.version_info.minor}. "
    "Run ./setup-py312.sh in a terminal, refresh the browser, and select the Python 3.12 kernel. "
    "See 00-start-here.ipynb for the steps."
)

In [ ]:
import sagemaker
import boto3
import time
import json
import os

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

In [ ]:
sm = boto3.client("sagemaker")
s3 = boto3.resource("s3")


def confirm(question):
    """Ask a yes/no question in the notebook and return True for y."""
    print(f"{question} (y/n)")
    return input().strip().lower() == "y"

---

## Delete the SageMaker projects

In [ ]:
# Get all projects created by the current domain
projects = [
    {"ProjectName":p["ProjectName"], "ProjectId":p["ProjectId"]} for p in sm.list_projects(MaxResults=100, SortBy="CreationTime")["ProjectSummaryList"] 
        if sm.describe_project(ProjectName=p["ProjectName"])["CreatedBy"]["DomainId"] == domain_id and p["ProjectStatus"] == "CreateCompleted"
]

print(f"These projects have been created by domain {domain_id}: {json.dumps(projects, indent=2)}")

In [ ]:
# Select projects to be deleted
projects_to_delete = []

for p in projects:
    print(f"Are you sure you want to delete this project: {p['ProjectName']}? (y/n)")
    choice = input()
    if choice == 'y':
        projects_to_delete.append(p)
        
print(f"***************************************")
print(f"The following projects will be deleted:\n{json.dumps(projects_to_delete, indent=2)}")
print(f"***************************************")

In [ ]:
for p in projects_to_delete:
    try:
        print(f"Deleting project {p['ProjectName']}:{sm.delete_project(ProjectName=p['ProjectName'])}")
    except Exception:
        pass

### Delete the monitoring schedules
Delete the monitoring schedules of the project endpoints, so that you can delete the endpoints.

In [ ]:
def delete_mon_schedules(endpoint_name, sm_client):
    print(f"Delete monitoring schedules for the endpoint: {endpoint_name}")
    for s in sm_client.list_monitoring_schedules(EndpointName=endpoint_name)["MonitoringScheduleSummaries"]:
        print(f"Delete monitoring schedule: {s['MonitoringScheduleName']}")
        r = sm_client.delete_monitoring_schedule(MonitoringScheduleName=s['MonitoringScheduleName'])
        print(r)

In [ ]:
for p in projects_to_delete:
    delete_mon_schedules(f"{p['ProjectName']}-staging", sm)
    delete_mon_schedules(f"{p['ProjectName']}-prod", sm)

### Delete the CloudFormation stacks of the endpoints
The deployment pipeline of each project created a CloudFormation stack for the staging endpoint and one for the production endpoint.

In [ ]:
cfn = boto3.client("cloudformation")

for p in projects_to_delete:
    for s in [
            f"sagemaker-{p['ProjectName']}-{p['ProjectId']}-deploy-staging",
            f"sagemaker-{p['ProjectName']}-{p['ProjectId']}-deploy-prod"
            ]:
        try:
            print(f"Delete CloudFormation stack: {s}")
            r = cfn.delete_stack(StackName=s)
            print(r)
            time.sleep(180)
        except Exception as e:
            print(f"Exception in deleting {s}:{e}")
            pass

### Delete the S3 buckets of the projects

<div class="alert alert-warning">The next cell deletes the S3 bucket of each deleted project. In an AWS-provided account you might not have the permissions to delete S3 buckets.</div>

In [ ]:
print(f"*****************************************************")
print(f"The following S3 buckets will be removed permanently!")
print(f"*****************************************************")
for p in projects_to_delete:
    print(f"sagemaker-project-{p['ProjectId']}")

In [ ]:
for p in projects_to_delete:
    !aws s3 rb s3://sagemaker-project-{p['ProjectId']} --force 

---

## Delete the inference endpoints

In [ ]:
endpoints = sm.list_endpoints()["Endpoints"]

In [ ]:
endpoints_to_delete = []

for ep in endpoints:
    print(f"Are you sure you want to delete this endpoint: {ep['EndpointName']}? (y/n)")
    choice = input()
    if choice == 'y':
        endpoints_to_delete.append(ep['EndpointName'])
        
print(f"*********** THESE ENDPOINTS WILL BE DELETED ***********")
print('\n'.join(endpoints_to_delete))
print(f"*******************************************************")

In [ ]:
for ep in endpoints_to_delete:
    try:
        delete_mon_schedules(ep, sm)
        time.sleep(10)
        print(f"Deleting the endpoint: {ep}:{sm.delete_endpoint(EndpointName=ep)}")
    except Exception as e:
        print(f"Exception in deleting {ep}:{e}")
        pass

---

## Delete the monitoring pipeline of step 4
The cells delete the EventBridge schedule, the SageMaker pipeline, and the SNS topic that notebook 04 created. If you ran the clean-up section of notebook 04, they're already deleted.

In [ ]:
monitoring_pipeline_name = "evidently-monitoring-pipeline"
monitoring_schedule_name = f"{monitoring_pipeline_name}-daily"
monitoring_topic_name = "sagemaker-evidently-drift-alerts"

scheduler = boto3.client("scheduler")
sns = boto3.client("sns")

if confirm(f"Delete the schedule {monitoring_schedule_name}, the pipeline {monitoring_pipeline_name}, and the SNS topic {monitoring_topic_name}?"):
    try:
        scheduler.delete_schedule(Name=monitoring_schedule_name)
        print(f"Deleted the schedule {monitoring_schedule_name}")
    except scheduler.exceptions.ResourceNotFoundException:
        print(f"The schedule {monitoring_schedule_name} doesn't exist")
    try:
        sm.delete_pipeline(PipelineName=monitoring_pipeline_name)
        print(f"Deleted the pipeline {monitoring_pipeline_name}")
    except sm.exceptions.ResourceNotFound:
        print(f"The pipeline {monitoring_pipeline_name} doesn't exist")
    topic_arn = f"arn:aws:sns:{region}:{boto3.client('sts').get_caller_identity()['Account']}:{monitoring_topic_name}"
    sns.delete_topic(TopicArn=topic_arn)  # succeeds also if the topic doesn't exist
    print(f"Deleted the SNS topic {monitoring_topic_name}")

---

## Delete the ML pipelines of step 3
The next cell lists the pipelines whose name starts with `from-idea-to-prod`. Earlier versions of notebook 03 created a new pipeline with a timestamp on each run.

In [ ]:
workshop_pipelines = [
    p["PipelineName"]
    for page in sm.get_paginator("list_pipelines").paginate(PipelineNamePrefix="from-idea-to-prod")
    for p in page["PipelineSummaries"]
]

for pipeline in workshop_pipelines:
    if confirm(f"Delete the pipeline {pipeline}?"):
        sm.delete_pipeline(PipelineName=pipeline)
        print(f"Deleted the pipeline {pipeline}")

---

## Delete the model package group
Notebooks 01, 02, and 03 registered model versions in one model package group. The next cell deletes all versions and then the group.

In [ ]:
try:
    model_package_group = target_model_package_group_name
except NameError:
    model_package_group = registered_model_name

model_packages = [
    p["ModelPackageArn"]
    for page in sm.get_paginator("list_model_packages").paginate(ModelPackageGroupName=model_package_group)
    for p in page["ModelPackageSummaryList"]
]

if confirm(f"Delete the model package group {model_package_group} with {len(model_packages)} model versions?"):
    for arn in model_packages:
        sm.delete_model_package(ModelPackageName=arn)
    sm.delete_model_package_group(ModelPackageGroupName=model_package_group)
    print(f"Deleted the model package group {model_package_group}")

---

## Delete the workshop data in S3

<div class="alert alert-warning">The next cells delete <b>all objects</b> under the workshop S3 prefix in the default SageMaker bucket. The MLflow artifacts are under the prefix <code>mlflow</code>, which the MLflow app deletion doesn't delete. Add it to the list if you don't need the artifacts anymore.</div>

In [ ]:
prefixes_to_delete = [
    bucket_prefix,
    #s3_fs_query_output_prefix
]

In [ ]:
print(f"************************************************************************")
print(f"All objects under the following S3 prefixes will be removed permanently!")
print(f"************************************************************************")
for p in prefixes_to_delete:
    print(f"{bucket_name}/{p}")

In [ ]:
for p in prefixes_to_delete:
    if confirm(f"Delete all objects under s3://{bucket_name}/{p}?"):
        s3.Bucket(bucket_name).objects.filter(Prefix=f"{p}/").delete()
        print(f"Deleted s3://{bucket_name}/{p}")

---

## Delete the project template and seed code of step 5
Notebook 05 uploaded the project template and the seed code to the default bucket, added the CORS policy to the bucket, and tagged the domain with the template location, so that Studio lists the template.

In [ ]:
template_keys = ["templates/deployment-only-project-template.yaml", "project-seed-code/deployment-seed-code.zip"]

if confirm(f"Delete the template and the seed code in s3://{bucket_name}, the bucket CORS policy, and the domain tag?"):
    for key in template_keys:
        s3.Object(bucket_name, key).delete()
        print(f"Deleted s3://{bucket_name}/{key}")
    boto3.client("s3").delete_bucket_cors(Bucket=bucket_name)
    print(f"Deleted the CORS policy of {bucket_name}")
    domain_arn = sm.describe_domain(DomainId=domain_id)["DomainArn"]
    sm.delete_tags(ResourceArn=domain_arn, TagKeys=["sagemaker:projectS3TemplatesLocation"])
    print(f"Deleted the tag sagemaker:projectS3TemplatesLocation of {domain_id}")

### Delete the IAM roles for SageMaker projects
Notebook 05 creates the CloudFormation stack `sagemaker-projects-roles-policies` with the IAM roles for SageMaker projects, if the roles didn't exist. Delete the stack only if no other project in the account uses these roles. In an AWS-provided account the roles belong to another stack: keep them.

In [ ]:
cfn = boto3.client("cloudformation")
roles_stack_name = "sagemaker-projects-roles-policies"

try:
    cfn.describe_stacks(StackName=roles_stack_name)
    if confirm(f"Delete the stack {roles_stack_name} with the IAM roles for SageMaker projects?"):
        cfn.delete_stack(StackName=roles_stack_name)
        print(f"Deleting the stack {roles_stack_name}")
except cfn.exceptions.ClientError:
    print(f"The stack {roles_stack_name} doesn't exist")

---

## Delete the SageMaker Python SDK configuration of step 3
Notebook 03 copied `config.yaml` with the defaults of the `@step` decorator to the user-level configuration folder of the SageMaker Python SDK. Delete it to go back to the SDK defaults.

In [ ]:
from platformdirs import user_config_dir

sdk_config = os.path.join(user_config_dir("sagemaker"), "config.yaml")
if os.path.exists(sdk_config) and confirm(f"Delete {sdk_config}?"):
    os.remove(sdk_config)
    print(f"Deleted {sdk_config}")

---

## Delete the MLflow app
Notebook [`00-start-here`](00-start-here.ipynb) created a managed MLflow app for the experiment tracking and stored its name in `mlflow_name`. The MLflow app costs money while it exists, so delete it when you don't need it anymore.

In [ ]:
# Delete the MLflow app created by the 00-start-here setup notebook.
# Its name was stored as `mlflow_name` (via %store), so we target exactly that
# app instead of hard-coding a name.
mlflow_apps_to_delete = []

try:
    mlflow_name
except NameError:
    print("`mlflow_name` not found in stored variables - did you run 00-start-here? Skipping MLflow app deletion.")
else:
    try:
        apps = sm.list_mlflow_apps()["Summaries"]
    except Exception as e:
        print(f"Unable to list MLflow apps: {e}")
        apps = []

    for app in apps:
        if app["Name"] != mlflow_name:
            continue
        # Skip apps that are already being deleted or have been deleted
        if app.get("Status") in ("Deleting", "Deleted", "DeleteFailed"):
            continue
        print(f"Are you sure you want to delete this MLflow app: {app['Name']} [{app.get('Status')}] ({app['Arn']})? (y/n)")
        choice = input()
        if choice == 'y':
            mlflow_apps_to_delete.append(app["Arn"])

print(f"*********** THESE MLFLOW APPS WILL BE DELETED ***********")
print('\n'.join(mlflow_apps_to_delete))
print(f"********************************************************")

In [ ]:
for arn in mlflow_apps_to_delete:
    try:
        print(f"Deleting the MLflow app: {arn}:{sm.delete_mlflow_app(Arn=arn)}")
    except Exception as e:
        print(f"Exception in deleting {arn}:{e}")
        pass

---

## Delete the container image in Amazon ECR
Notebook [`00-start-here`](00-start-here.ipynb) built the XGBoost container image and pushed it to the Amazon ECR repository `sagemaker-xgboost-byoc`. ECR storage costs money, so delete the repository when you don't need the image anymore.

<div class="alert alert-warning">The next cells delete the ECR repository with <b>all</b> its images. Make sure that no endpoint or job still uses the image.</div>

In [ ]:
ecr = boto3.client("ecr")

# Default repository name created by the setup notebook
ecr_repo_name = "sagemaker-xgboost-byoc"

# Prefer the repository derived from the image URI stored by the setup notebook
try:
    ecr_repo_name = xgboost_image.split("/")[-1].split(":")[0]
    print(f"Derived ECR repository from stored xgboost_image: {ecr_repo_name}")
except NameError:
    print(f"xgboost_image not found in stored variables, using default repo name: {ecr_repo_name}")

In [ ]:
print(f"Are you sure you want to delete the ECR repository '{ecr_repo_name}' and ALL images in it? (y/n)")
choice = input()
if choice == 'y':
    try:
        r = ecr.delete_repository(repositoryName=ecr_repo_name, force=True)
        print(f"Deleted ECR repository {ecr_repo_name}:{r}")
    except ecr.exceptions.RepositoryNotFoundException:
        print(f"ECR repository {ecr_repo_name} not found - nothing to delete")
    except Exception as e:
        print(f"Exception in deleting repository {ecr_repo_name}:{e}")
else:
    print("Skipped ECR repository deletion")

---

## Delete the feature groups
Only needed if you ran the optional notebook `additional-topics/03c-feature-store`.

In [ ]:
feature_groups = sm.list_feature_groups(
    FeatureGroupStatusEquals="Created", 
    SortOrder="Descending", 
    SortBy="CreationTime"
)["FeatureGroupSummaries"]

In [ ]:
feature_groups

In [ ]:
# Select feature groups to be deleted
feature_groups_to_delete = []

for fg in feature_groups:
    print(f"Are you sure you want to delete this feature group: {fg['FeatureGroupName']}? (y/n)")
    choice = input()
    if choice == 'y':
        feature_groups_to_delete.append(fg["FeatureGroupName"])
        
print(f"********************************************")
print(f"The following feature groups will be deleted:\n{json.dumps(feature_groups_to_delete, indent=2)}")
print(f"********************************************")

In [ ]:
def delete_offline_store(feature_group_name: str):
    try:
        offline_store_config = sm.describe_feature_group(FeatureGroupName=feature_group_name)['OfflineStoreConfig']

    except Exception:
        print(f'Feature group: {feature_group_name} does NOT have an offline store!')
        return
    
    offline_store_s3_uri = offline_store_config['S3StorageConfig']['ResolvedOutputS3Uri']
    print(f"all feature store objects under {offline_store_s3_uri} will be deleted!")
    print("Are you sure you want to these objects ? (y/n)")
    
    choice = input()
    if choice == 'y':
        !aws s3 rm {offline_store_s3_uri} --recursive

<div class="alert alert-info"> 💡 <strong> The following code cell will delete the selected feature groups!</strong>
</div>

In [ ]:
for fg in feature_groups_to_delete:
    print(f"Deleting the feature group: {fg}")
    delete_offline_store(fg)
    sm.delete_feature_group(FeatureGroupName=fg)

## Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>